# 05 - Machine Learning: Unpaid Tagging Classification
**Objective:** build and evaluate classifiers for the tag (1 = tagged) using **Data All (Q1-Q4) only**.
**Data Validasi is not loaded in this notebook**; it is used only in Phase 7, after the model and threshold are fixed.

**Principles:** preprocessing is fitted only on training data (inside a Pipeline, also inside every CV fold); model selection uses
cross-validation on the training set; the test set is used once for reporting; accuracy is never the only metric; the threshold is chosen
on out-of-fold training predictions with a stated business assumption. Associations are not causes: feature importance is not causal.
Outputs are cleared before committing; no customer-level predictions are saved.

In [ ]:
import sys
sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.tree import export_text
import joblib

from src.preprocessing import load_clean, ID_COL, TARGET
from src.feature_engineering import build_features, WINDOW_ALL, get_feature_set, detect_constant_features
from src.modeling import RANDOM_STATE, base_models, tune_model, out_of_fold_proba, make_pipeline
from src.evaluation import (evaluate, threshold_metrics, threshold_sweep, fbeta_column, plot_roc_pr, plot_confusion,
                            plot_calibration, plot_threshold_tradeoff)

sns.set_style("whitegrid"); pd.set_option("display.max_columns", 40); pd.set_option("display.width", 170)
FIG, MODELS, REPORTS = "../figures", "../models", "../reports"

clean_all = load_clean("../data/raw/finanku_data_all.csv")
F = build_features(clean_all, WINDOW_ALL)
y = clean_all.set_index(ID_COL)[TARGET]
print("Customers:", len(F), "| tagged:", int(y.sum()), f"({y.mean():.2%})")

## 1. Target, class balance and leakage checks
Tagged customers are 26.7% of the data: mildly imbalanced. We keep the natural class mix (no resampling) and evaluate with PR-AUC, recall, precision and a tuned threshold instead of accuracy.

**Two feature sets** (Phase 2): `no_card` (primary, conservative) and `with_card` (adds `card_start`). The card indicator is
perfectly tied to the tag in Data All (all 611 customers without a card are tagged), so results with it are a sensitivity check, not the main result.

In [ ]:
FEATURE_SETS = {name: get_feature_set(name) for name in ["no_card", "with_card"]}
const = detect_constant_features(F)
for name, feats in FEATURE_SETS.items():
    assert ID_COL not in feats and TARGET not in feats, "identifier or target in features"
    assert not set(feats) & set(const), f"constant feature in {name}: {set(feats) & set(const)}"
    assert all(f in F.columns for f in feats)
    print(f"{name}: {len(feats)} features OK (no ID, no target, no constant columns)")
print("\nConstant columns in engineered data (excluded from both sets):", const)
display(y.value_counts().rename("customers").to_frame().assign(share=lambda d: d["customers"] / d["customers"].sum()).round(4))

## 2. Stratified train/test split
80% training (model selection by CV), 20% test (kept aside, used once for reporting).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(F, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"train: {len(X_train):,} (tagged {y_train.mean():.2%}) | test: {len(X_test):,} (tagged {y_test.mean():.2%})")

## 3. Baselines and alternative models (stratified 5-fold CV on the training set)
Baselines: **DummyClassifier** (always predicts the class prior) and **Logistic Regression**. Alternatives: **Decision Tree, Random Forest, Gradient Boosting**.
Small hyper-parameter grids are searched by ROC-AUC. Preprocessing (median imputation, scaling, one-hot encoding, constant-feature removal) is inside the pipeline.

In [ ]:
import time
tuned = {}          # (feature_set, model) -> fitted GridSearchCV
rows = []
for fs, feats in FEATURE_SETS.items():
    for name in base_models():
        t0 = time.time()
        gs = tune_model(name, X_train, y_train, feats)
        tuned[(fs, name)] = gs
        i = gs.best_index_
        rows.append({"feature_set": fs, "model": name, "cv_roc_auc_mean": gs.best_score_,
                     "cv_roc_auc_std": gs.cv_results_["std_test_score"][i],
                     "best_params": {k.replace("model__", ""): v for k, v in gs.best_params_.items()}})
        print(f"{fs:10s} {name:22s} CV AUC {gs.best_score_:.3f}  ({time.time()-t0:.0f}s)")
cv_table = pd.DataFrame(rows)
display(cv_table.round(4))

## 4. Model selection (one-standard-error rule)
Among models within one standard error of the best CV ROC-AUC, we choose the **simplest** (order: Logistic regression < Decision tree < Random forest < Gradient boosting).
Rationale: more complex models that are statistically indistinguishable add opacity without evidence of benefit.

In [ ]:
COMPLEXITY = ["Dummy (class prior)", "Logistic regression", "Decision tree", "Random forest", "Gradient boosting"]
selected = {}
for fs in FEATURE_SETS:
    sub = cv_table[cv_table.feature_set == fs].set_index("model")
    best = sub["cv_roc_auc_mean"].idxmax()
    se = sub.loc[best, "cv_roc_auc_std"] / np.sqrt(5)
    eligible = [m for m in COMPLEXITY if sub.loc[m, "cv_roc_auc_mean"] >= sub.loc[best, "cv_roc_auc_mean"] - se]
    selected[fs] = eligible[0]
    print(f"{fs}: best CV model = {best} ({sub.loc[best,'cv_roc_auc_mean']:.3f}, SE {se:.3f}) -> selected (simplest within 1 SE): {selected[fs]}")

fig, ax = plt.subplots(figsize=(8, 3.8))
plot_df = cv_table.copy()
for k, (fs, color) in enumerate([("no_card", "#2E5E8C"), ("with_card", "#D97B29")]):
    s = plot_df[plot_df.feature_set == fs]
    ax.errorbar(np.arange(len(s)) + (k - 0.5) * 0.2, s["cv_roc_auc_mean"], yerr=s["cv_roc_auc_std"], fmt="o", color=color, capsize=3, label=fs)
ax.set_xticks(range(len(COMPLEXITY))); ax.set_xticklabels(COMPLEXITY, rotation=20, ha="right", fontsize=8)
ax.axhline(0.5, ls="--", color="grey"); ax.set_ylabel("CV ROC-AUC (mean +/- sd)"); ax.legend(fontsize=8)
ax.set_title("Cross-validated ROC-AUC by model and feature set", fontweight="bold")
fig.tight_layout(); fig.savefig(f"{FIG}/13_cv_model_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Test-set evaluation (all tuned models, default threshold 0.50)
Metrics: ROC-AUC, PR-AUC (baseline = prevalence 26.7%), Brier score, expected calibration error, and threshold-based accuracy, precision, recall, specificity, F1, balanced accuracy.
Note the difference between **ranking quality** (AUC) and **decisions at 0.50**: with 27% prevalence a threshold of 0.50 flags few customers, so recall can be very low even if AUC is decent.

In [ ]:
test_rows, test_proba = [], {}
for (fs, name), gs in tuned.items():
    p = gs.predict_proba(X_test[FEATURE_SETS[fs]])[:, 1]
    test_proba[(fs, name)] = p
    test_rows.append({"feature_set": fs, "model": name, **evaluate(y_test, p, 0.5)})
test_table = pd.DataFrame(test_rows)
cols = ["feature_set", "model", "roc_auc", "pr_auc", "brier", "ece", "accuracy", "precision", "recall", "specificity", "f1", "balanced_accuracy"]
display(test_table[cols].round(3))

In [ ]:
# Curves for the primary feature set (no card information)
primary = {m: test_proba[("no_card", m)] for m in COMPLEXITY}
fig = plot_roc_pr(y_test, primary, f"{FIG}/14_roc_pr_no_card.png"); plt.show()
fig = plot_calibration(y_test, {m: test_proba[("no_card", m)] for m in ["Logistic regression", "Decision tree", "Random forest", "Gradient boosting"]},
                       f"{FIG}/15_calibration_no_card.png"); plt.show()

In [ ]:
sel_name = selected["no_card"]
m_default = threshold_metrics(y_test, test_proba[("no_card", sel_name)], 0.5)
fig = plot_confusion(m_default, f"{sel_name}, threshold 0.50 (test)", f"{FIG}/16a_confusion_default.png"); plt.show()
print({k: round(v, 3) if isinstance(v, float) else v for k, v in m_default.items()})

## 6. Threshold optimisation (selected primary model)
The default 0.50 is not automatically sensible. The threshold is chosen on **out-of-fold predictions on the training set** (the test set is not touched), using an explicit assumption:

- **Assumption (illustrative, no cost data available):** failing to flag a tagged customer (false negative) is more costly than reviewing an extra customer (false positive), so recall is weighted twice as much as precision (**F2**).
- **Capacity constraint (hypothetical):** at most **40% of customers** can be flagged for review.
- **Rule:** maximise F2 among thresholds that flag at most 40% of customers.

Both numbers are assumptions that a real organisation would replace with its own costs and capacity.

In [ ]:
BETA, CAPACITY = 2.0, 0.40
feats = FEATURE_SETS["no_card"]
best_pipe = tuned[("no_card", sel_name)].best_estimator_
oof = out_of_fold_proba(best_pipe, X_train, y_train, feats)

sweep = threshold_sweep(y_train, oof)
sweep["f2"] = fbeta_column(sweep, BETA)
eligible = sweep[sweep["flagged_share"] <= CAPACITY]
chosen = float(eligible.loc[eligible["f2"].idxmax(), "threshold"])
print(f"Chosen threshold = {chosen:.2f}  (max F2 with flagged share <= {CAPACITY:.0%}, out-of-fold training data)")

show = sweep[sweep["threshold"].isin([0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, chosen])].drop_duplicates("threshold")
display(show[["threshold", "precision", "recall", "f1", "f2", "flagged_share", "fp", "fn"]].round(3))
fig = plot_threshold_tradeoff(sweep, chosen, f"{FIG}/17_threshold_tradeoff.png"); plt.show()

In [ ]:
p_test = test_proba[("no_card", sel_name)]
m_chosen = threshold_metrics(y_test, p_test, chosen)
fig = plot_confusion(m_chosen, f"{sel_name}, threshold {chosen:.2f} (test)", f"{FIG}/16b_confusion_chosen.png"); plt.show()

comp = pd.DataFrame({"threshold 0.50": threshold_metrics(y_test, p_test, 0.5), f"chosen {chosen:.2f}": m_chosen}).T
display(comp[["accuracy", "precision", "recall", "specificity", "f1", "balanced_accuracy", "flagged_share", "tp", "fp", "fn", "tn"]].round(3))
n = len(y_test)
print(f"At threshold {chosen:.2f}: {m_chosen['flagged_share']*n:.0f} of {n:,} test customers flagged ({m_chosen['flagged_share']:.1%}); "
      f"{m_chosen['tp']} of {m_chosen['tp']+m_chosen['fn']} tagged customers found; {m_chosen['fp']} flagged customers were non-tagged.")
print("Operational reading per 1,000 customers flagged:", round(1000*m_chosen['precision']), "tagged,", round(1000*(1-m_chosen['precision'])), "non-tagged.")

**False positives vs false negatives:** a false positive is a non-tagged customer flagged (extra review workload, possible unnecessary contact); a false negative is a tagged customer that is missed
(the pattern goes unnoticed). What these cost depends on the unknown meaning of the tag, which is why the threshold rationale above is an explicit assumption.

## 7. Explainability
1. Logistic regression coefficients (linear baseline; limited because relationships are non-linear).
2. Selected model's rules / feature importance and **permutation importance** on the test set (drop in ROC-AUC when a feature is shuffled).
3. SHAP if available. **Importance is not causality:** it shows what the model uses to separate tags in this data, not what drives tagging.

In [ ]:
lr = tuned[("no_card", "Logistic regression")].best_estimator_
names = lr[:-1].get_feature_names_out()
coef = pd.Series(lr.named_steps["model"].coef_[0], index=names).sort_values(key=abs, ascending=False)
odds = pd.DataFrame({"coefficient": coef, "odds_ratio_per_1SD_or_dummy": np.exp(coef)}).head(12)
display(odds.round(3))

In [ ]:
print(f"Selected model: {sel_name}")
if sel_name == "Decision tree":
    tree = best_pipe.named_steps["model"]
    print(export_text(tree, feature_names=list(best_pipe[:-1].get_feature_names_out()), max_depth=4, show_weights=True))   # weights = [non-tagged, tagged] customers in the leaf

pi = permutation_importance(best_pipe, X_test[feats], y_test, scoring="roc_auc", n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"feature": feats, "auc_drop_mean": pi.importances_mean, "auc_drop_std": pi.importances_std}).sort_values("auc_drop_mean", ascending=False)
display(imp.head(10).round(4))

top = imp.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(top["feature"], top["auc_drop_mean"], xerr=top["auc_drop_std"], color="#2E5E8C")
ax.set_xlabel("Drop in test ROC-AUC when shuffled"); ax.set_title(f"Permutation importance ({sel_name}, no card information)", fontweight="bold")
fig.tight_layout(); fig.savefig(f"{FIG}/18_permutation_importance.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
try:
    import shap
    Xt = best_pipe[:-1].transform(X_test[feats])
    Xt = Xt.toarray() if hasattr(Xt, "toarray") else np.asarray(Xt)
    fn = list(best_pipe[:-1].get_feature_names_out())
    explainer = shap.TreeExplainer(best_pipe.named_steps["model"])
    sv = explainer.shap_values(Xt)
    sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if np.ndim(sv) == 3 else sv)
    shap.summary_plot(sv, Xt, feature_names=fn, show=False, max_display=10)
    plt.gcf().savefig(f"{FIG}/19_shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()
except Exception as e:      # SHAP is optional (it may not support the newest Python versions)
    print("SHAP skipped:", type(e).__name__, "-", str(e)[:150])
    print("Permutation importance and model rules above already explain the model.")

## 8. Feature-set sensitivity: with vs without card information
`with_card` adds `card_start`, which is tied to the tag (see Phase 3/5). Compare test performance to see how much of the performance depends on it.

In [ ]:
cmp = test_table[test_table.model.isin(["Logistic regression", "Decision tree", "Random forest", "Gradient boosting"])]
display(cmp.pivot(index="model", columns="feature_set", values=["roc_auc", "pr_auc"]).round(3))

## 9. Save artefacts
The fitted model is saved locally (`models/*.joblib` is git-ignored). Only **aggregate** metrics are saved to `reports/`; no customer-level predictions are written.

In [ ]:
import os
os.makedirs(MODELS, exist_ok=True)
artifact = {"pipeline": best_pipe, "features": feats, "feature_set": "no_card", "model": sel_name, "threshold": chosen,
            "threshold_rule": f"max F{BETA:g} with flagged share <= {CAPACITY:.0%} (out-of-fold, training data)",
            "trained_on": "Data All (Q1-Q4), 80% stratified training split"}
joblib.dump(artifact, f"{MODELS}/final_model_no_card.joblib")

summary = test_table[cols].merge(cv_table[["feature_set", "model", "cv_roc_auc_mean", "cv_roc_auc_std"]], on=["feature_set", "model"])
summary.to_csv(f"{REPORTS}/ml_results_summary.csv", index=False)
print("Saved model artefact and reports/ml_results_summary.csv (aggregate metrics only)")
print(f"Primary model: {sel_name}, feature set no_card, threshold {chosen:.2f}")

## Checklist before Phase 7
- Data Validasi untouched; model, feature set and threshold are fixed above.
- Primary result uses `no_card`; `with_card` is a sensitivity result.
- Summarise findings in `reports/ml_findings.md` after reviewing the outputs.